In [1]:
from pathlib import Path

DATA = Path("../HW2/data/data.txt")

# repr 而不是 print:行尾的空格和 \n 只有 repr 看得见
with DATA.open(encoding="utf-8") as fh:
    for _, line in zip(range(3), fh):
        print(repr(line))

'2004-03-31 03:38:15.757551 2 1 122.153 -3.91901 11.04 2.03397\n'
'2004-02-28 00:59:16.02785 3 1 19.9884 37.0933 45.08 2.69964\n'
'2004-02-28 01:03:16.33393 11 1 19.3024 38.4629 45.08 2.68742\n'


In [2]:
from collections import Counter

single, collapsed = Counter(), Counter()
with DATA.open(encoding="utf-8", errors="replace") as fh:
    for line in fh:
        line = line.rstrip("\r\n")          # 只去换行,不去空格
        single[len(line.split(" "))] += 1   # 按单个空格切
        collapsed[len(line.split())] += 1   # 折叠连续空白后切

print("总行数    ", f"{sum(single.values()):,}")
print("split(' ')", dict(sorted(single.items())))
print("split()   ", dict(sorted(collapsed.items())))

总行数     2,313,682
split(' ') {8: 2313682}
split()    {3: 526, 5: 373, 6: 4, 7: 92976, 8: 2219803}


In [3]:
ROSTER_FILE = Path("../HW2/data/mote_locs.txt")

# 每行 'moteid x y',我们现在只要第一列
roster = {int(l.split()[0]) for l in ROSTER_FILE.read_text().splitlines() if l.strip()}
print(len(roster), min(roster), max(roster))

54 1 54


In [4]:
n = Counter()
with DATA.open(encoding="utf-8", errors="replace") as fh:
    for line in fh:
        f = line.rstrip("\r\n").split(" ")     # 格 2 已证明恒为 8 个字段
        n["rows"] += 1
        if not f[3].isdigit():                 # moteid 为空
            n["no_mote"] += 1
            continue
        if int(f[3]) not in roster:            # 发了数据但不在名单上
            n["off_roster"] += 1
            continue
        n["accepted"] += 1
        blank = f[4:8].count("")               # 这次传输缺了几个 channel
        n["readings"] += 4 - blank             # long 格式下展开成几条 reading

for k, v in n.items():
    print(f"{k:<12}{v:>12,}")

rows           2,313,682
accepted       2,303,290
readings       9,119,212
off_roster         9,866
no_mote              526


In [5]:
CH = ("temperature", "humidity", "light", "voltage")
stat = {c: [float("inf"), float("-inf")] for c in CH}

with DATA.open(encoding="utf-8", errors="replace") as fh:
    for line in fh:
        f = line.rstrip("\r\n").split(" ")
        if not f[3].isdigit() or int(f[3]) not in roster:
            continue
        for name, cell in zip(CH, f[4:8]):      # 空字段跳过,缺席不是 0
            if cell:
                v = float(cell)
                s = stat[name]
                s[0] = min(s[0], v)
                s[1] = max(s[1], v)

for c in CH:
    print(f"{c:<12}{stat[c][0]:>16,.4f}{stat[c][1]:>16,.4f}")

temperature         -38.4000        385.5680
humidity         -8,983.1300        137.5120
light                 0.0000      1,847.3600
voltage               0.0091         18.5600


In [6]:
SAMPLE = Path("sample.txt")

with DATA.open(encoding="utf-8", errors="replace") as src, SAMPLE.open("w", encoding="utf-8") as dst:
    for _, line in zip(range(300_000), src):
        dst.write(line)

print(f"{SAMPLE.stat().st_size / 1e6:.1f} MB")

19.7 MB


In [7]:
Path("sql").mkdir(exist_ok=True)

In [8]:
import sqlite3

DB = Path("lab.db")

def connect(path=DB):
    conn = sqlite3.connect(path, isolation_level=None)   # autocommit,BEGIN/COMMIT 由我们自己放
    conn.execute("PRAGMA foreign_keys = ON")             # per-connection,默认关
    return conn

DB.unlink(missing_ok=True)                               # 每次从 data.txt 重建,不原地改
conn = connect()
conn.executescript(Path("sql/schema.sql").read_text(encoding="utf-8"))

roster_rows = []
for entry in ROSTER_FILE.read_text().splitlines():
    if entry.strip():
        mote, x, y = entry.split()                       # 'moteid x y'
        roster_rows.append((int(mote), float(x), float(y)))

conn.execute("BEGIN")
conn.executemany("INSERT INTO sensors (sensor_id, x_m, y_m) VALUES (?,?,?)", roster_rows)
conn.execute("COMMIT")

print("sensors :", conn.execute("SELECT count(*) FROM sensors").fetchone()[0])
print("readings:", conn.execute("SELECT count(*) FROM readings").fetchone()[0])

sensors : 54
readings: 0


In [9]:
try:
    conn.execute("INSERT INTO readings VALUES (99, '2004-01-01 00:00:00.000000', 0.0, 1, 'temperature', 1.0)")
    print("没拦住 —— PRAGMA 没生效")
except sqlite3.IntegrityError as e:
    print("拦住了:", e)

拦住了: FOREIGN KEY constraint failed
